# GEMS G11 — Recovery-first master Colab execution

**Process once, save, then train.** Expensive geophysical and DEM features are computed once, verified, saved
permanently on Drive as a versioned dataset (`03_features/datasets/<feature_version>/`) and reused by every later
experiment. On every launch the notebook:

1. mounts Drive and checks the code revision;
2. **inspects and verifies** the existing feature artifacts (nothing is deleted or overwritten);
3. computes **only** the missing / invalid layers (DEM statistics chunk by chunk, each chunk committed to Drive);
4. finalizes and validates the dataset (`COMPLETE.json` only after validation);
5. on later launches skips extraction entirely;
6. builds geographically separated splits that pass hard integrity gates;
7. trains, predicts, evaluates and exports the official GeoTIFF (never uploaded).

Every completed stage survives a Colab disconnect: re-run sections 00–03, then continue where it stopped.
DTI values are the repository **offline DTI proxy**, not the official leaderboard DTI.

## 00 — Configuration

Edit only this cell. Unknown keys and incompatible values are rejected in section 03.

In [ ]:
CONFIG = {
    "repo": "mybenkhadda/GEMS---drivendata",
    "ref": "g11-colab-production",          # branch / tag / commit that contains the recovery-first G11 code
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "repo_root": "/content/GEMS",
    "local_root": "/content/g11_work",       # local SSD scratch (tile cache, chunk staging, feature cache)

    # FULL_RESUME | RECOVER_ONLY | PROCESS_FEATURES_ONLY | FINALIZE_FEATURES_ONLY | SPLITS_ONLY | TRAIN_ONLY |
    # INFER_ONLY | AUDIT_ONLY | INGEST_ONLY | EDA_ONLY | EVALUATE_ONLY | SUBMISSION_ONLY | SMOKE_TEST
    "run_mode": "FULL_RESUME",
    "resume": True,
    "force_recompute": False,                # downstream caches only - never recomputes verified features

    "feature_policy": "REUSE_VERIFIED",      # REUSE_VERIFIED | RESUME_MISSING | FORCE_NEW_VERSION (never default)
    "feature_version": "g11_features_v1",
    "split_version": "v2",                   # v1 (the interrupted run's split) is marked INVALID once v2 passes
    "recovery_verify": "AUTO",               # AUTO | FULL (re-read + re-hash every layer)
    "save_every_chunk": True,
    "feature_time_budget_min": 0.0,          # >0: stop DEM chunk processing after N minutes (resume next launch)

    "dem_statistics_resolution_m": 10.0,     # new DEM feature version at 10 m; the 2 m layers stay their own version
    "dem_native_resolution_m": 1.0,          # targeted fault profiles keep the native 1 m DEM
    "dem_training_resolution_m": 25.0,
    "geo_resolution_m": 100.0,

    "use_gpu": True, "mixed_precision": True, "seed": 2026,
    "run_audit": True, "run_ingestion": True, "run_features": True, "run_eda": True, "run_splits": True,
    "run_training": True, "run_fusion": True, "run_candidates": True, "run_evaluation": True, "run_submission": True,
    "models": ["xgboost", "segformer", "gated_fusion", "segment_ranker"],

    "training_target_buffer_m": 200.0,
    "unlabeled_background_weight": 0.3,
    "strict_integrity": True,
    "failure_policy": "STOP_REQUIRED_SKIP_OPTIONAL",
    "save_intermediate": True, "generate_figures": True, "generate_html_report": True,
    "official_submission": True,
    "upload_submission": False,              # must stay False: G11 never uploads
    "smoke_window_km": 30.0,
}

## 01 — Drive mount

Mounts Drive and reports GPU, RAM and local disk.

In [ ]:
import os, sys, time, json, shutil, subprocess, re
from pathlib import Path
T_START = time.time()
from google.colab import drive
drive.mount("/content/drive")
if not Path(CONFIG["drive_root"]).is_dir():
    raise RuntimeError(f"Drive root {CONFIG['drive_root']} not found: check the path or remount "
                       "(drive.mount('/content/drive', force_remount=True))")
Path(CONFIG["local_root"]).mkdir(parents=True, exist_ok=True)
def sh(*args, check=True, cwd=None):
    r = subprocess.run(list(args), capture_output=True, text=True, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError(f"command failed: {' '.join(args)}\n{r.stderr[-2000:]}")
    return r.stdout.strip()
print(sh("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader", check=False) or "no GPU")
print("RAM:", sh("free", "-g", check=False).splitlines()[1] if shutil.which("free") else "?")
print("local disk free (GB):", round(shutil.disk_usage("/content").free / 1e9, 1))
OUT_ROOT = Path(CONFIG["drive_root"]) / "outputs/G11_MULTISCALE_FAULT_DISCOVERY"
print("existing G11 output root:", OUT_ROOT, "exists:", OUT_ROOT.is_dir())

## 02 — Repository checkout

Clones `repo_root` if absent and checks out `ref` detached. A checkout with local modifications is **not**
overwritten (the cell stops; never a blind `git pull`). If `ref` is not on GitHub or lacks the recovery-first
modules, the cell stops with the deployment commands.

In [ ]:
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"
REF, ROOT = CONFIG["ref"], Path(CONFIG["repo_root"])
DEPLOY_HELP = f"""
The recovery-first G11 code is not available at {REPO_URL} @ {REF}.
On the machine with the working copy:
    git checkout {REF}            # (git checkout -b {REF} if the branch does not exist yet)
    git add gems_bench/g11 configs/g11_multiscale.yaml docs/g11 notebooks/colab/GEMS_G11_*.ipynb
    git commit -m "G11 recovery-first pipeline"
    git push -u origin {REF}
then re-run this cell (or set CONFIG['ref'] to a branch / commit that contains the files below).
"""
is_sha = re.fullmatch(r"[0-9a-f]{7,40}", REF) is not None
remote = sh("git", "ls-remote", "--heads", "--tags", REPO_URL, REF, check=False)
if not remote and not is_sha:
    raise RuntimeError(DEPLOY_HELP)
if not (ROOT / ".git").exists():
    sh("git", "clone", "--filter=blob:none", REPO_URL, str(ROOT))
dirty = sh("git", "status", "--porcelain", "--untracked-files=no", cwd=str(ROOT))
if dirty:
    raise RuntimeError(f"{ROOT} has local modifications - not overwriting them:\n{dirty}\n"
                       "commit / stash them, or set CONFIG['repo_root'] to a fresh path")
sh("git", "fetch", "--depth", "1", "origin", REF, cwd=str(ROOT))
sh("git", "checkout", "--detach", "FETCH_HEAD", cwd=str(ROOT))
COMMIT = sh("git", "rev-parse", "HEAD", cwd=str(ROOT))
REQUIRED = ["gems_bench/g11/pipeline.py", "gems_bench/g11/recovery.py", "gems_bench/g11/dem_engine.py",
            "gems_bench/g11/feature_dataset.py", "gems_bench/g11/splits_v2.py", "gems_bench/g11/training_data.py",
            "gems_bench/g11/notebook_config.py", "configs/g11_multiscale.yaml"]
missing = [f for f in REQUIRED if not (ROOT / f).is_file()]
if missing:
    raise RuntimeError(f"{REF} ({COMMIT[:10]}) lacks {missing}." + DEPLOY_HELP)
sh(sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[colab]", "xgboost", "lightgbm", "psutil", "pyarrow")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import gems_bench.g11 as G11
from gems_bench.g11 import pipeline as G11P, notebook_config as NC, colab_runtime as CR
print("G11", G11.__version__, "| commit", COMMIT)

## 03 — Environment validation

Synthetic integrity suite (LEVEL 1) must pass; then CONFIG is validated, adapted to the YAML, and the runtime configured (only batch size / AMP / workers change; every adjustment is recorded).

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "gems_bench/g11/tests"), "-q", "-p", "no:cacheprovider",
                    "-x", "--ignore", str(ROOT / "gems_bench/g11/tests/test_real_data.py")],
                   capture_output=True, text=True, cwd=str(ROOT))
print(r.stdout[-2500:])
if r.returncode != 0:
    raise RuntimeError("LEVEL 1 synthetic tests failed - stop (see output above)")
from gems_bench.g11.config import load_config
import pandas as pd
from IPython.display import Image, Markdown
BASE = load_config()
PLAN = NC.build_plan(CONFIG, BASE)
CFG = load_config(overrides=PLAN.overrides)
INFO = CR.detect_runtime(CONFIG["drive_root"], CONFIG["local_root"])
CFG, ADJUSTMENTS = CR.configure_runtime(CFG, INFO, CONFIG["use_gpu"], CONFIG["mixed_precision"])
P = G11P.G11Pipeline(CFG)
RUN = NC.StepRunner(P, PLAN)
CR.save_runtime_report(P.ctx.d("runtime") / "runtime_report.json", INFO, ADJUSTMENTS, P.tracker)
print("adjustments:", ADJUSTMENTS, "| notes:", PLAN.notes)
print("output root:", P.ctx.root, "| feature version:", P.feature_version, "| policy:", P.feature_policy,
      "| run namespace:", P.ns)
display(PLAN.table())
AUD = RUN("audit", P.audit, check=lambda s: None if s["grid"]["epsg"] == 32611 else "master grid is not EPSG:32611")
ING = RUN("ingest", P.ingest, check=lambda s: None if s.get("fault_pixels", 1) > 0 else "no mapped fault pixel")
print(json.dumps(json.loads((P.ctx.d("ingest") / "coverage_report.json").read_text()), indent=1))

## 04 — Existing feature recovery

Discovers every feature file, reads its real header (shape, dtype, data offset), checks size / truncation, content
(finite fraction vs the value recorded when it was written; nodata), provenance (stage manifest, source hashes,
registry hash, write time) and configuration (layer produced by the current config; feature-relevant config sections
equal to those of the commit that wrote it). Classes: VERIFIED_COMPLETE · RECOVERABLE (manifest reconstructed) ·
INCOMPLETE · INCOMPATIBLE · MISSING. First launch reads each legacy layer once (~30 GB); later launches only `stat`
the files and re-hash a sample.

In [ ]:
REC = RUN("recover", P.recover)
if REC is not None:
    print(json.dumps(REC.as_dict(), indent=1, default=str)[:4000])
    display(REC.table.groupby(["family", "status"]).size().unstack(fill_value=0))
    bad = REC.table[REC.table.status.isin(["INCOMPLETE", "INCOMPATIBLE"])]
    if len(bad): display(bad[["layer_id", "relpath", "status", "reason"]].head(30))
    display(Image(str(P.ctx.root / "recovery" / "feature_coverage_map.png")))

## 05 — Missing-chunk processing

Only layers that are not verified are computed. DEM statistics use the fast engine: every usable 1 m tile is read
**once** into a local 10 m tile cache, the grid is processed in 12.8 km chunks, each chunk is validated locally,
committed to Drive through a verified temporary file and given a completion sidecar. Interrupted? Re-run: committed
chunks are skipped. 2 m and 10 m layers never share an id, a file or a manifest.

In [ ]:
budget = CONFIG["feature_time_budget_min"] * 60 or None
PF = RUN("process_features", lambda: P.process_features(max_seconds=budget),
         check=lambda s: None if s["complete"] else "feature processing not finished - re-run this cell to resume")
print(json.dumps(PF, indent=1, default=str))

## 06 — Feature dataset finalization

Writes `03_features/datasets/<feature_version>/` (schema, cubes, label schema, valid / DEM-availability masks, availability, native-window index, fault geometry registry, processing config, provenance) and `COMPLETE.json` only after validation. Verified layers of the other DEM resolution (legacy 2 m) are kept as their own version. Split assignments are never part of a feature dataset.

In [ ]:
DSM = RUN("finalize_features", P.finalize_features)
if DSM is not None:
    print(DSM)
    ds_dir = P.ctx.d("features") / "datasets"
    print("datasets:", sorted(p.name for p in ds_dir.iterdir() if (p / "COMPLETE.json").is_file()))

## 07 — Feature integrity tests

QUICK (every file / header / size, sampled sha256, schema, grid, masks, DEM coverage explicitly missing) — FULL re-hashes everything (`feature_policy='RESUME_MISSING'`).

In [ ]:
VAL = RUN("validate_features", P.validate_features, check=lambda r: None if r.passed else str(r.problems))
if VAL is not None: display(VAL.checks)
display(pd.DataFrame([P.feature_stats()]).T)

## 08 — EDA using saved features

Computed from the finalized dataset only (no raw-feature recomputation).

In [ ]:
EDA = RUN("eda", P.eda)
d = P.ctx.d("eda")
if (d / "eda_summary.md").is_file():
    display(Markdown((d / "eda_summary.md").read_text()[:15000]))
    for f in ["fig_coverage.png", "fig_contrast.png", "fig_utility.png"]:
        if (d / f).is_file(): display(Image(str(d / f)))

## 09 — Geographic split reconstruction

Split v2: 20 km blocks, small fault systems kept whole, compact regions, seeded assignments balancing area, mapped
faults and DEM coverage after the embargo. The old split (train 5 065 055 / validation 5 317 / stack 318 / test 0 px)
is marked INVALID only after v2 passes every gate; its files are kept.

In [ ]:
SPL = RUN("splits", P.splits)
if SPL is not None:
    print(json.dumps(SPL, indent=1))
    display(Image(str(P.ctx.d("splits", P.split_version) / "split_map.png")))

## 10 — Split integrity tests

Every partition non-empty, positives, independent blocks, embargo, patch feasibility, small-fault-system leakage, no target-derived predictor, no final-test tuning. Training is refused unless all pass.

In [ ]:
def _gates():
    sd = P.ctx.d("splits", P.split_version)
    g = pd.read_parquet(sd / "split_gates.parquet")
    lk = json.loads((sd / "leakage_report.json").read_text())
    display(g)
    return g, lk
VS = RUN("validate_splits", _gates, check=lambda t: None if t[0].passed.all() and t[1]["passed"] else "split gate failed")

## 11 — Training dataset preparation

Persisted sample indices per split, train-only scalers, class weights, sampling configuration (memory-mapped features, no copy of the cube).

In [ ]:
def _prep():
    out = {}
    for e in [x for x in PLAN.experiments if x in ("E0", "E1")]:
        if e == "E1" and not any(n.startswith("dem100_") for n in P.store.reg.name):
            continue
        td = P.training_data(e)
        out[e] = {"dir": str(td.dir), "samples": {k: len(v) for k, v in td.indices.items()},
                  "channels": len(td.channels), "class_weights": td.class_weights}
    return out
TDS = RUN("training_data", _prep)
print(json.dumps(TDS, indent=1, default=str))

## 12 — XGBoost training (E0 geophysics, E1 + DEM summaries)

In [ ]:
X = [e for e in PLAN.experiments if e in ("E0", "E1")]
TXGB = RUN("train_pixel", lambda: P.train(X), show={"experiments": X, "target": CFG["labels"]["training_target"]})
display(TXGB)

## 13 — SegFormer training (E2 + loss ablation)

Checkpoint every epoch in its own experiment directory on Drive; resume from `last.pt` of the SAME configuration only (other configurations are archived, never erased).

In [ ]:
TSEG = RUN("train_segformer", lambda: P.train(["E2"]), show={"loss": CFG["training"]["loss"],
           "ablation": CFG["training"].get("loss_ablation"), "batch": CFG["training"]["batch_size"]})
display(TSEG)

## 14 — Multiresolution fusion training (E3 + mode ablation)

100 m geophysics + DEM patches with modality / availability masks (local 25 m DEM mosaic; no Drive reads per patch).

In [ ]:
TGF = RUN("train_gated", lambda: P.train(["E3"]), show={"mode": CFG["models"]["gated_fusion"]["mode"],
          "ablation": CFG["models"]["gated_fusion"].get("mode_ablation")})
display(TGF)

## 15 — Segment ranker training (E4)

Needs the validation-selected pixel map: (1) every trained map present (regenerated from saved models if missing),
(2) pixel ensemble E5 fitted on stack / selected on validation, (3) validation leaderboard + `selection.json`
(before any test metric), (4) candidates from the selected calibrated map and label-free geophysical lineaments →
linking → deduplication → candidate DEM morphology → LightGBM binary vs lambdarank chosen on validation.

In [ ]:
def _ranker():
    inv = P.predict_all()
    display(inv)
    fu = P.fusion()
    display(fu)
    lb = P.evaluate(final=False)
    return P.infer()
RANK = RUN("train_ranker", _ranker, check=lambda m: None if m["n_candidates"] > 0 else "no candidate")
print(RANK)

## 16 — Full-grid inference

Inventory of every probability raster (present / regenerated) and the selected calibrated probability map.

In [ ]:
INF = RUN("inference", P.predict_all)
if INF is not None: display(INF)
sp = P.d("predictions") / "selected_calibrated_probability.tif"
print("selected calibrated map:", sp, sp.is_file())

## 17 — Candidate recovery

Ranked, UNVERIFIED candidate fault segments evaluated at fixed predicted-length budgets (validation offline DTI proxy).

In [ ]:
def _cands():
    d = P.d("infer")
    display(pd.read_parquet(d / "ranker_validation.parquet"))
    bt = pd.read_parquet(d / "budget_validation.parquet")
    display(bt)
    return json.loads((d / "inference_manifest.json").read_text())
CR_ = RUN("candidates", _cands)
print(CR_)

## 18 — Offline evaluation

Validation leaderboard, then the held-out test ONCE (refused without the validation selection). The offline DTI proxy is **not** the official DTI.

In [ ]:
FIN = RUN("evaluate", lambda: P.evaluate(final=True))
if FIN is not None:
    cols = ["experiment_id", "split", "model", "PR_AUC", "PR_AUC_ci_low", "PR_AUC_ci_high", "ROC_AUC", "candidate_recall",
            "predicted_length_km", "fragmentation", "offline_DTI_proxy", "training_time_s", "peak_GPU_memory_gb"]
    display(FIN[[c for c in cols if c in FIN]].sort_values(["split", "PR_AUC"], ascending=[True, False]))

## 19 — Official submission GeoTIFF

Single band on the competition template grid, written with the repository's validated writer, reopened and validated, SHA-256 + manifest. `official_DTI = unavailable` unless an official evaluator is installed. **Not uploaded.**

In [ ]:
SUB = RUN("submit", P.submit, check=lambda r: None if r["validation"]["passed"] else "GeoTIFF validation failed")
if SUB is not None:
    print(json.dumps({k: SUB[k] for k in ("path", "sha256", "validation", "official_DTI", "uploaded")}, indent=1, default=str))

## 20 — Final report

In [ ]:
REP = RUN("report", lambda: P.report(PLAN, INFO))
if REP is not None:
    display(Markdown(Path(REP["paths"]["md"]).read_text()[:20000]))
    print("HTML:", REP["paths"]["html"], "| leaderboard:", REP["paths"]["leaderboard"])
os.sync()
print("step results:", json.dumps(RUN.results, indent=1))
print(f"total notebook time: {(time.time() - T_START) / 3600:.2f} h | outputs: {P.ctx.root}")